# Viewing SyMTRS samples

Port of the generator's `view_dataset` notebook onto `symtrs view`. Per transform it shows the day RGB (chosen resolution), the no-shadow and night RGB (2048 only), the colourised mask with legend, and depth in metres (EXR is centimetres; turbo_r, 1-99 percentile clip).

Layout: `<root>/<split>/<modality>[_lr/x{2048//res}]/{rgb_day|rgb_night|rgb_noshadow|seg_day|depth_day}_{res}_<key>.{png|exr}`. Mask grey = class id x 63 (background, building, green, road, car).

Needs `numpy pillow matplotlib` and `OpenEXR` (`pip install -e '.[exr]'`). The same thing from a shell: `symtrs view --data-root ROOT --split val --index 0 --save out.png`.

In [ ]:
%matplotlib inline
from pathlib import Path

from symtrs.commands import view

DATA_ROOT = Path(r"/data/SyMTRS")  # edit me
for split in view.SPLITS:
    try:
        print(f"{split:5s}: {len(view.list_transforms(DATA_ROOT, split))} transforms")
    except FileNotFoundError as exc:
        print(f"{split:5s}: {exc}")

## One transform at native resolution

`build_figure` takes any file of the transform and finds its siblings. A missing file raises `FileNotFoundError` naming it, rather than drawing a blank panel.

In [ ]:
sample = view.list_transforms(DATA_ROOT, "val")[0]
fig = view.build_figure(sample)

## The same transform at 512

`res` names the pixel size of the data. Night and no-shadow stay at 2048: they exist nowhere else.

In [ ]:
fig = view.build_figure(sample, res=512)

## Class balance of a split

Reads masks only (no images), so it is cheap enough to run on a sample of a split.

In [ ]:
import numpy as np
from PIL import Image

total = np.zeros(len(view.CLASS_NAMES))
files = view.list_transforms(DATA_ROOT, "train")[:20]
for f in files:
    with Image.open(view.sibling(f, "seg", "day", 256)) as im:
        ids, _ = view.seg_ids(np.array(im))
    total += np.array(view.class_percentages(ids))
for name, pct in zip(view.CLASS_NAMES, total / len(files)):
    print(f"{name:11s} {pct:5.1f}%")